# 02 — Indicatorii de acces

Pornim de la `data/processed/set_analiza.csv`: 3.180 de localități, cu populația și
cu 33 de coloane de unități sanitare.

Construim **două** indicatoare, pe două axe:
- **îngrijire** — localitatea are medic de familie?
- **medicație** — localitatea are farmacie?

Combinate, dau patru situații diferite, iar ele sunt răspunsul la întrebarea de business.

## 1. Încarcă datele

Un singur lucru de reținut: `dtype={"cod_siruta": "str"}` la citire.
Fără el, codul devine număr și join-urile viitoare se rup.

In [12]:
import pandas as pd
df = pd.read_csv("../data/processed/set_analiza.csv", dtype={"cod_siruta": "str"})

df

,Judete,cod_siruta,localitate,65-69 ani,70-74 ani,75-79 ani,80-84 ani,85 ani si peste,Total,populatie_65plus,...,Puncte farmaceutice,Sanatorii TBC,Sanatorii balneare,Societate civila medicala de specialitate,Societate medicala civila,Societate stomatologica civila medicala,Spitale,Unitati medico-sociale,Unitati sanitare (inclusiv unitati asimilate spitalelor) numai cu internare de zi,total_unitati
0,Alba,1017,MUNICIPIUL ALBA IULIA,5082,4639,2698,1227,994,74137,14640,...,0,0,0,0,0,0,2,0,1,260
1,Alba,1071,CIUGUD,179,208,136,74,54,3448,651,...,0,0,0,0,0,0,0,0,0,6
2,Alba,1151,ORAS ABRUD,331,296,179,110,87,4829,1003,...,0,0,0,0,0,0,1,0,0,22
3,Alba,1213,MUNICIPIUL AIUD,1736,1686,1022,510,477,23600,5431,...,0,0,0,0,0,0,2,0,0,67
4,Alba,1348,MUNICIPIUL BLAJ,1239,1221,824,454,359,19821,4097,...,0,0,0,0,0,0,1,0,0,51
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3175,Vrancea,178929,BILIESTI,141,131,92,56,63,2434,483,...,0,0,0,0,0,0,0,0,0,2
3176,Vrancea,178938,GOLOGANU,207,159,149,109,116,2833,740,...,0,0,0,0,0,0,0,0,0,2
3177,Vrancea,178947,OBREJITA,77,77,62,37,42,1542,295,...,0,0,0,0,0,0,0,0,0,0
3178,Vrancea,178956,RASTOACA,156,139,118,79,89,2259,581,...,0,0,0,0,0,0,0,0,0,2


## 2. Uită-te la ce ai

Câte rânduri, ce coloane. Caută-le pe cele două care te interesează —
scrie `[c for c in df.columns if "armaci" in c]` ca să le vezi pe cele cu farmacii.

In [21]:
[c for c in df.columns if "armac" in c]


['Depozite farmaceutice', 'Farmacii', 'Puncte farmaceutice']

## 3. Primul indicator: `are_medic_familie`

O coloană cu `True` / `False`: localitatea are cel puțin un cabinet de medicină de familie.

Ai făcut deja ceva asemănător la pasul 6 din notebook-ul 01, când ai scris
`df["pondere_65plus"] = ...`. Aceeași formă: partea stângă e o coloană nouă în `df`,
partea dreaptă e o expresie pe o coloană existentă.

Ce e nou: comparația `> 0` pe o coloană produce direct `True`/`False` pentru fiecare rând.

Verifică după: `df["are_medic_familie"].sum()` — câte `True`? Aștept **2.821**.

In [ ]:
df['are_medic_familie'] = df['Cabinete medicale de familie'] > 0

In [16]:
df['are_medic_familie'].sum()

np.int64(2821)

## 4. Al doilea indicator: `are_farmacie`

Aici ai o decizie de luat, nu doar de scris cod.

Există **două** coloane: `Farmacii` și `Puncte farmaceutice`. Un punct farmaceutic e o
formă redusă de farmacie, permisă în localități mici tocmai pentru că nu susțin o farmacie
completă.

- doar `Farmacii` → 2.037 localități
- `Farmacii` **sau** `Puncte farmaceutice` → 2.554 localități

Peste 500 de localități depind de alegerea ta. Pentru o analiză despre acces la
medicație în mediul rural, care variantă crezi că e corectă — și de ce?

Scrie indicatorul, apoi scrie motivul în README.

In [25]:
df['are_farmacie'] = (df['Farmacii'] > 0) | (df['Puncte farmaceutice'] > 0)
df['are_farmacie'].sum()

np.int64(2553)

## 5. Cele patru situații

Acum combină-le. `pd.crosstab(df["are_medic_familie"], df["are_farmacie"])` îți dă
un tabel 2×2 cu numărul de localități din fiecare categorie.

Uită-te la colțul cu `False` / `False`. Alea sunt comunele fără nimic.

In [27]:
pd.crosstab(df["are_medic_familie"], df["are_farmacie"])

are_farmacie,False,True
are_medic_familie,,
False,169,190
True,458,2363


## 6. Cine sunt

Pentru fiecare din cele patru grupe, care e ponderea medie de 65+?

Indiciu: `df.groupby([...])["pondere_65plus"].mean()`.

Și apoi: listează primele 15 localități fără medic **și** fără farmacie, ordonate
după ponderea vârstnicilor. Aia e lista scurtă către care lucrezi de trei săptămâni.

In [28]:
df.groupby(["are_farmacie", "are_medic_familie"])["pondere_65plus"].mean()

are_farmacie  are_medic_familie
False         False                0.236058
              True                 0.211674
True          False                0.193260
              True                 0.187851
Name: pondere_65plus, dtype: float64

In [38]:
masca = (~df["are_medic_familie"]) & (~df["are_farmacie"])

In [39]:
filtru = df[masca]


In [40]:
filtru.nlargest(15, "pondere_65plus")[["Judete", "localitate", "Total", "populatie_65plus", "pondere_65plus"]]

,Judete,localitate,Total,populatie_65plus,pondere_65plus
1588,Hunedoara,BATRANA,109,51,0.467890
744,Buzau,PARDOSI,287,128,0.445993
1599,Hunedoara,CERBAL,388,171,0.440722
1597,Hunedoara,BUNILA,309,128,0.414239
854,Caras-Severin,BREBU NOU,328,135,0.411585
1624,Hunedoara,TOMESTI,1001,394,0.393606
2781,Teleorman,UDA-CLOCOCIOV,1149,441,0.383812
714,Buzau,CHILIILE,397,151,0.380353
742,Buzau,ODAILE,596,218,0.365772
1608,Hunedoara,LELESE,327,119,0.363914


In [44]:
masca2 = (~df["are_medic_familie"]) & (~df["are_farmacie"] & (df['Total'] >= 1000))
filtru2 = df[masca2]
ordonat = filtru2.sort_values("pondere_65plus", ascending=False)
limitat = ordonat.groupby("Judete").head(2)
final = limitat.head(10)
final[["Judete", "localitate", "Total", "populatie_65plus", "pondere_65plus"]]

,Judete,localitate,Total,populatie_65plus,pondere_65plus
1624,Hunedoara,TOMESTI,1001,394,0.393606
2781,Teleorman,UDA-CLOCOCIOV,1149,441,0.383812
2777,Teleorman,FANTANELE,1248,426,0.341346
2990,Valcea,PIETRARI,2891,971,0.335870
1956,Mehedinti,OBARSIA DE CAMP,1502,458,0.304927
886,Caras-Severin,NAIDAS,1077,322,0.298979
1611,Hunedoara,MARTINESTI,1008,297,0.294643
3004,Valcea,STROESTI,2532,743,0.293444
1426,Giurgiu,ISVOARELE,1334,384,0.287856
965,Cluj,MANASTIRENI,1263,362,0.286619
